# Which of these worlds could have liquid water — and why does your test reject Earth?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/02_liquid_water.ipynb).

Astronomers have found six thousand planets around other stars. For most of them we know three numbers and nothing else: how hot the star is, how big it is, and how far out the planet orbits. Today you turn those three numbers into a temperature, and a temperature into a verdict.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## The question

Liquid water is the one thing every living system we know of needs. So the first question anyone
asks about a newly found planet is whether it could hold any — and the honest answer usually has
to come from almost nothing. Nobody has photographed the surface of a planet around another star.
For most of the six thousand we have found, the archive holds the star's temperature, the star's
size, and the width of the planet's orbit.

That is enough to compute something: the temperature a planet would sit at if starlight were the
only thing heating it. Today you write that calculation as a **function**, run it over three
thousand worlds with a **loop**, and let an **if** statement sort them into ones that could hold
liquid water and ones that could not.

Then you run the same test on Earth, whose answer we already know. **Which of these worlds could
have liquid water — and why does your test reject Earth?**

## What you'll be able to do

**The science.** Say what a planet's equilibrium temperature is, and compute it from three numbers
an archive will give you for almost any planet. Measure the difference between that temperature and
the real one, on the three worlds where we have both. And say what that difference is made of.

**The code.** `for` loops over a list and over `range(n)` · the accumulator pattern, with
`list.append` · `if` / `elif` / `else` and the comparison operators · `and`, `or`, `not` · `None`,
and why you cannot compare it with a number · `abs` · writing your own functions with `def`,
arguments, `return` and a docstring · `help` · and marking a plot up with `plt.axvline` and
`plt.text`.

**Eight places where you write something: five in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it. The first two homework parts ask for numbers and then
for a sentence about them, so those have a second cell as well.

1. What temperature would starlight alone hold a world at?
2. Does that test say Venus could have liquid water?
3. How much of a world's warmth comes from its air, not its star?
4. Which of three thousand real planets pass — and what does the archive not know about them?

## Setup

Run the next cell once. You are not expected to follow it — it is here so that everything below it
is short.

**Coming later:** it uses **pandas**, the tables library we meet properly in the tables week, to
fetch the NASA Exoplanet Archive's list of confirmed planets and hand it back as six ordinary
**lists** — one per column, all in the same order, so position 40 of each list is the same planet.
Lists and loops are this week's subject; pandas is not.

The archive is a live catalogue: it grows, and rows get revised. Every count printed in this
notebook came from the copy stored with the course, so if you run it live and get a slightly
different number, that is the archive having moved, not you having made a mistake.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load():
    """the NASA Exoplanet Archive's confirmed planets: live if the network is up, the copy stored with the course if not"""
    try:
        return pd.read_csv("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
                           "select+pl_name,st_teff,st_rad,pl_orbsmax,pl_rade,pl_eqt"
                           "+from+ps+where+default_flag=1&format=csv")
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + "week02_exoplanets.csv")

archive = load()
archive = archive.astype(object).where(archive.notna(), None)


def column(name):
    """one column of the archive as an ordinary list, with None wherever it has no value"""
    return list(archive[name])


planet_names = column("pl_name")     # what the planet is called
star_temps = column("st_teff")       # its star's surface temperature, in K
star_radii = column("st_rad")        # its star's radius, in radii of the Sun
distances = column("pl_orbsmax")     # the width of the planet's orbit, in AU
planet_radii = column("pl_rade")     # the planet's radius, in radii of the Earth
archive_temps = column("pl_eqt")     # the archive's own equilibrium temperature, where it has one

print("planets in the archive:", len(planet_names))

## 1. What temperature would starlight alone hold a world at?

Before three thousand planets, three. Venus, Earth and Mars are the three rocky worlds we have
measurements from, and they are the only planets anywhere whose surface temperature we know from
having been there.

The numbers below are from NASA's NSSDC planetary fact sheets. Those sheets are no longer on the
live web — `nssdc.gsfc.nasa.gov/planetary/factsheet/` redirects to a NASA landing page that does
not carry them — so what is cited here is the last archived copy: the Internet Archive's capture
of [the Earth sheet](https://web.archive.org/web/20250820/https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html),
[Venus](https://web.archive.org/web/20250820/https://nssdc.gsfc.nasa.gov/planetary/factsheet/venusfact.html)
and [Mars](https://web.archive.org/web/20250820/https://nssdc.gsfc.nasa.gov/planetary/factsheet/marsfact.html),
taken **2025-08-20**, read 2026-08-31. Those links open; the original address does not.

**Bond albedo** is the fraction of sunlight a world reflects, and Earth's is the one number here
that needs an argument, because the sheet does not agree with itself. It prints a Bond albedo of
0.294 — and, four rows down, a black-body temperature of 254.0 K. Those two cannot both be right:
0.294 gives 255.1 K, and only 0.306 reproduces the 254.0 the sheet itself prints. This notebook
takes 0.306, the value that makes the page self-consistent, and once you have the formula and the
albedo term — section 3 — you can check that for yourself. Every temperature below moves if you
change it, which is why the value, where it came from and when it was read all have to be written
down.

Five lists, in the same order — Venus, Earth, Mars, going outwards from the Sun. A **`for` loop**
does the same thing once for each item in a list.

In [ ]:
worlds = ["Venus", "Earth", "Mars"]
sun_distances = [0.723, 1.000, 1.524]      # width of the orbit, in AU
bond_albedos = [0.770, 0.306, 0.250]       # fraction of sunlight reflected
surface_temps = [737.0, 288.0, 214.0]      # measured surface temperature, in K
air_pressures = [92.0, 1.014, 0.006]       # atmospheric pressure at the surface, in bar

for world in worlds:
    print(world)

`world` is a name the loop invents for you: on the first pass it holds `"Venus"`, on the second
`"Earth"`, on the third `"Mars"`. The indented lines run once per item.

That form is fine when you need one list. With five lists you need the **position** instead, so
that you can read the same position out of all of them — the trick from last week, where the
position of the largest magnitude also found the place name. `range(n)` gives you the whole run of
positions: `range(3)` is 0, 1, 2. And `len(worlds)` is 3, so `range(len(worlds))` is every position
in the list without your having to count them.

In [ ]:
for i in range(len(worlds)):
    print(worlds[i], "orbits at", sun_distances[i], "AU and its surface sits at",
          surface_temps[i], "K under", air_pressures[i], "bar of air")

Three lines of output, and the third column already carries a puzzle: Venus's air is 92 bar against
Earth's 1.014, and Mars's is 0.006. Hold on to that.

The other thing a loop is for is building something up as it goes. **Set a counter to 0 before the loop starts and add to it inside; same idea for a list — start
it empty and append inside.** `results.append(x)` adds one item to the end of the list `results`.

### ✏️ Your turn 1

Sunlight thins out with distance: a planet twice as far from its star catches a
quarter as much. So the sunlight reaching a world, compared with Earth, is `1 / distance ** 2` —
where `**` means "to the power of", so `distance ** 2` is the distance squared.

Start an empty list called `sunlight`, loop over the three worlds, and append each one's share of
sunlight compared with Earth, rounded to two decimal places with `round(value, 2)`. Print the list
when the loop has finished.

**Use these names**: `sunlight`.

In [ ]:
# ← your answer here


A planet sits in a rain of starlight. Some of that light bounces straight off; the rest is absorbed
and warms the planet up. A warm planet glows — in the infrared, not visibly — and the warmer it
gets the harder it glows, until it is losing exactly as much heat as it is catching. The
temperature at which those two rates balance is the planet's **equilibrium temperature**.

**Equilibrium temperature: the temperature a planet would sit at if starlight were the only thing heating it and it had
no air.**

Balancing those two rates gives one formula. It is standard — deriving it needs the
Stefan–Boltzmann law, which is not today's business — and it needs three numbers about the star and
the orbit, plus the fraction of light that bounces off:

```
temperature = star_temp * (star_radius / (2 * distance)) ** 0.5 * (1 - albedo) ** 0.25
```

- `star_temp` is the star's surface temperature in kelvin;
- `star_radius / distance` is how big the star looks from the planet, with both lengths in the
  same units; the extra `2` is not part of that view. It is there because the planet catches
  starlight on one circular face but glows from its whole surface — four times the area — so the
  heat it caught is spread over day and night alike;
- `** 0.5` is a square root and `** 0.25` a fourth root, so `(1 - albedo) ** 0.25` is the fourth
  root of the fraction of light the planet keeps.

For the Sun the two numbers we need are its surface temperature, 5,772 K, and its radius,
0.00465047 AU — the IAU's 2015 nominal solar values (Resolution B3), checked 2026-08-31. Start with
Earth, and with `albedo` left out entirely, as if the planet reflected nothing at all.

In [ ]:
sun_temp = 5772             # K, the Sun's surface temperature
sun_radius_au = 0.00465047  # the Sun's radius, in AU

earth_temp = sun_temp * (sun_radius_au / (2 * 1.000)) ** 0.5
print("Earth, reflecting nothing and with no air:", round(earth_temp, 1), "K")

### ✏️ Your turn 2

Do the same for all three worlds. Loop over the positions in `worlds`, compute
each one's temperature the way the cell above did — reflecting nothing, `sun_distances[i]` in place
of the `1.000` — and print the world's name and its temperature rounded to one decimal place.

Nothing here is new; it is the range loop from section 1 with the arithmetic from the cell above
dropped inside it.

In [ ]:
# ← your answer here


## 2. Does that test say Venus could have liquid water?

Now turn a temperature into a verdict. Water is liquid between its melting point and its boiling
point, and at Earth's sea-level pressure those are 273 K and 373 K. That is the window this
notebook will use throughout. It is a convention, and a shaky one: both ends move with pressure,
and on Mars's 0.006 bar of air water boils only a few degrees above freezing. We are adopting it
anyway, and noting what we adopted.

An **`if` statement** runs its indented block only when a comparison comes out true. `elif` (short
for "else if") offers another comparison to try if the first failed, and `else` catches everything
left. The comparisons themselves are `<`, `>`, `<=`, `>=`, `==` for "is equal to" and `!=` for "is
not". You can chain two of them: `273 <= temperature <= 373` is true when the temperature is inside
the window, which is shorter than saying it twice.

Here is the whole test, over the three worlds, with a counter accumulating alongside it.

In [ ]:
in_window = 0
for i in range(len(worlds)):
    temperature = sun_temp * (sun_radius_au / (2 * sun_distances[i])) ** 0.5
    if temperature < 273:
        verdict = "too cold"
    elif temperature > 373:
        verdict = "too hot"
    else:
        verdict = "liquid water possible"
        in_window = in_window + 1
    print(worlds[i], round(temperature, 1), "K -", verdict)

print("worlds the test accepts:", in_window)

## 3. How much of a world's warmth comes from its air, not its star?

Two of the three pass. One of the two is Earth, which is reassuring. The other is Venus — whose
surface temperature, in the first table of this notebook, is 737 K. That is 364 degrees above the
boiling point of water, and our test just said liquid water was possible there. Something is
missing, and section 1 already put it on screen: 92 bar of air.

The formula also has a term we have not used yet.

**Albedo: the fraction of the starlight falling on a world that it reflects straight back into space; only
the rest is absorbed and turned into heat.**

A world that reflects more of its starlight keeps less of it and runs cooler, and the three are
very different: Mars reflects a quarter of what reaches it, Earth 0.306 of it, and Venus — wrapped
in cloud from pole to pole — 0.770. Those are the `bond_albedos` from section 1, and putting them
into the formula means multiplying by `(1 - bond_albedos[i]) ** 0.25`.

### Predict before you run

Write a number down before you run the next cell. With each world's real albedo in the formula, how
many of the three end up inside the 273–373 K window: **0, 1, 2 or 3?** Venus reflects far more
than Earth does, so Venus should cool by more than Earth does. Commit to an answer, out loud, to
whoever is sitting next to you.

In [ ]:
albedo_temps = []
for i in range(len(worlds)):
    starlight = sun_temp * (sun_radius_au / (2 * sun_distances[i])) ** 0.5
    albedo_temps.append(starlight * (1 - bond_albedos[i]) ** 0.25)
    print(worlds[i], round(albedo_temps[i], 1), "K")

None of them. Venus falls to 226.7 K and Earth to 254.0 K — nineteen degrees below freezing — and
Mars was never in. Run the test one way and it accepts Earth and Venus together; run it the other
way and it rejects Earth, Venus and Mars alike. It has not been right once.

So how wrong is it? We know what these three worlds are actually like.

### ✏️ Your turn 3

Loop over the three worlds and print, for each, its measured surface temperature
from `surface_temps`, the temperature the test just gave it from `albedo_temps`, the difference
between them rounded to one decimal place, and its `air_pressures` value. Four numbers on a line,
and the third one is `surface_temps[i] - albedo_temps[i]`.

**Use these names**: `gap`, for the difference.

In [ ]:
# ← your answer here


Mars, 0.006 bar of air: the test is out by 4.2 K. Earth, 1.014 bar: out by 34.0 K. Venus, 92 bar:
out by 510.3 K.

Same physics, same formula, three thicknesses of air, and the error tracks the air. That
difference has a name.

**The greenhouse effect: the difference between the temperature a world would have with no air and the temperature it
actually has.**

It is worth being clear about what just happened, because it is the opposite of how this usually
gets taught. Nobody defined the greenhouse effect and then went looking for it. We computed what
starlight alone would do, compared that with three thermometers, and the leftovers came out at 4.2,
34.0 and 510.3 K — in the order of how much air each world has. Mars to Venus is four orders of
magnitude in pressure, 0.006 bar to 92, and it comes out as two orders of magnitude in the
leftover. The greenhouse effect is that leftover, measured.

The figure below plots it. The diagonal is where a world with no air would sit, and each planet
stands above it by exactly the amount its own air adds — Mars by 4.2 K, so close to the line you
have to look for the gap, Venus by 510.3 K, which is almost the whole height of the plot. All
three sit to the left of the window.

In [ ]:
plt.plot([195, 390], [195, 390], color="0.6", lw=1)     # where a world with no air would sit
plt.scatter(albedo_temps, surface_temps, s=45)
plt.axvline(273, color="0.8")
plt.axvline(373, color="0.8")
for i in range(len(worlds)):
    plt.text(albedo_temps[i] + 4, surface_temps[i], worlds[i])
plt.text(285, 720, "liquid water window")
plt.text(330, 305, "no air at all")
plt.xlabel("equilibrium temperature, at the world's own albedo (K)")
plt.ylabel("measured surface temperature (K)")
plt.title(f"Starlight alone against reality (n = {len(worlds)})")
plt.show()

## 4. Which of three thousand real planets pass — and what does the archive not know about them?

Three thousand of them are sitting in the setup cell, and running the test over all of them is a
loop you can already write. First, one piece of housekeeping. You have now typed that formula four
times, and typing it a fifth time inside that loop would mean any correction had to be made in
five places.

A **function** is the fix: write the recipe once, give it a name, and from then on ask for it by
name. `def` starts the definition, the names in brackets are what the function needs to be given,
and `return` hands one value back to whoever called it. The triple-quoted line just inside is a
**docstring**, and it says what the function is for.

In [ ]:
def equilibrium_temperature(star_temp, star_radius, distance, albedo):
    """the temperature starlight alone would hold a planet at, in kelvin"""
    return star_temp * (star_radius * sun_radius_au / (2 * distance)) ** 0.5 * (1 - albedo) ** 0.25


print("Earth at its own albedo:", round(equilibrium_temperature(5772, 1.0, 1.000, 0.306), 1), "K")
help(equilibrium_temperature)

Two things changed on the way in. `star_radius` is now measured in radii of the Sun, because that
is the unit the archive uses — the function multiplies by `sun_radius_au` itself, so the Sun goes
in as `1.0`. And `albedo` is an argument rather than something baked in, so the same function
answers the albedo-0 question and the albedo-0.306 question without being rewritten.

`help(equilibrium_temperature)` printed the docstring back. That is what writing one buys you.

A function can also hand back **two** values at once: `return temperature, verdict` gives both, and
`t, v = check_planet(...)` catches them in that order.

### ✏️ Your turn 4

Write a function called `check_planet` that takes the same four arguments —
`star_temp`, `star_radius`, `distance`, `albedo` — and returns two things: the temperature, and a
verdict string that is `"too cold"` below 273 K, `"too hot"` above 373 K, and
`"liquid water possible"` in between.

Give it a docstring. Inside it, call `equilibrium_temperature` rather than retyping the formula,
then use the same `if` / `elif` / `else` as section 2. Finish by calling it on Venus twice — once
with albedo `0.0` and once with albedo `0.770` — and printing both results.

**Use these names**: `check_planet`.

In [ ]:
# ← your answer here


The setup cell loaded the NASA Exoplanet Archive: every confirmed planet around another star, one
row each. Six lists, all in the same order.

Most of those rows are incomplete, and that is the normal condition of an astronomical catalogue
rather than a flaw in it. Different discovery methods measure different things: a planet found by
watching it cross its star gives a radius, a planet found by watching its star wobble gives a mass,
and neither gives both. Where the archive has no value, the setup cell put **`None`** in the list —
Python's word for "nothing here".

`None` is not zero and it is not a small number: it is the absence of one, and Python refuses to
compare it with a number at all. `None < 1.6` is an error, not a `False`. That refusal is a gift.
The test for it is `is None`, or `is not None` for the other way round, and `and` joins two
conditions so that both have to hold.

The cell below keeps only the planets that carry all three numbers the formula needs.

In [ ]:
usable_names = []
usable_star_temps = []
usable_star_radii = []
usable_distances = []
usable_radii = []
usable_archive_temps = []
for i in range(len(planet_names)):
    if star_temps[i] is not None and star_radii[i] is not None and distances[i] is not None:
        usable_names.append(planet_names[i])
        usable_star_temps.append(star_temps[i])
        usable_star_radii.append(star_radii[i])
        usable_distances.append(distances[i])
        usable_radii.append(planet_radii[i])
        usable_archive_temps.append(archive_temps[i])

print("planets with all three numbers:", len(usable_names), "out of", len(planet_names))

More than half the archive is gone before we start, and no amount of cleverness gets it back: those
planets were never measured that way. (Six appends to keep six lists in step is also the clearest
possible argument for the tables library we meet in a few weeks, which does this in one line.)

Before trusting our own arithmetic on three thousand worlds, check it against somebody else's. The
archive publishes its own equilibrium temperature, `pl_eqt`, for some planets — computed by whoever
wrote the discovery paper, with their own assumptions. Where both exist we can compare. `abs(x)`
gives the size of a difference without caring which way round it went, and the last three lines of
the loop are the other thing an accumulator does: hold on to the biggest one seen so far.

In [ ]:
ours = []
theirs = []
close = 0
above = 0
below = 0
worst_gap = 0
worst_planet = ""
worst_distance = 0
for i in range(len(usable_names)):
    if usable_archive_temps[i] is not None:
        temperature = equilibrium_temperature(usable_star_temps[i], usable_star_radii[i],
                                              usable_distances[i], 0.0)
        gap = abs(temperature - usable_archive_temps[i])
        ours.append(temperature)
        theirs.append(usable_archive_temps[i])
        if gap < 10:
            close = close + 1
        elif temperature > usable_archive_temps[i]:
            above = above + 1
        else:
            below = below + 1
        if gap > worst_gap:
            worst_gap = gap
            worst_planet = usable_names[i]
            worst_distance = usable_distances[i]

print("planets we can compare against:", len(ours))
print("of those, within 10 K of the published value:", close)
print("the rest, ours hotter than theirs:", above, " ours colder:", below)
print("the largest disagreement:", round(worst_gap), "K, on", worst_planet, "-",
      worst_distance, "AU from its star")

In [ ]:
plt.scatter(theirs, ours, s=3)
plt.plot([0, 4000], [0, 4000], color="0.6", lw=1)       # where perfect agreement would sit
plt.xlabel("the archive's published equilibrium temperature (K)")
plt.ylabel("our equilibrium temperature, albedo 0 (K)")
plt.title(f"Our arithmetic against the archive's (n = {len(ours)})")
plt.show()

952 of 1525 agree within 10 K, and the cloud hugs the diagonal, so this is the calculation the
field uses. The loop counted the rest rather than leaving it to the eye: of the 573 that miss by
10 K or more, **456 sit above the line and 117 below**. The 456 are our own doing — we let every
planet absorb all the light it catches, and the discovery papers each used an albedo of their own,
so our answers run hot.

The 117 below the line are a different animal, and they are the ones your eye goes to, because they
run far off the diagonal instead of crowding beside it. The worst of them says what they are.
HD 284149 AB b orbits 431 AU out — four hundred times the width of Earth's orbit — and at that
distance the starlight our formula is built on has almost nothing left in it, so our answer
comes out near zero and the two disagree by 2379 K. The archive's number for that planet is not a
response to its star at all: a young planet photographed that far out is still glowing with the
heat of its own formation. The formula is not wrong about it. It is answering a question that does
not apply.

Agreement with the professionals is not agreement with reality, either. Everyone in that figure is
computing the same quantity, and section 3 already showed what that quantity leaves out.

One more number, and it is a radius. A planet's temperature says nothing about whether it has a
surface: at about 1.6 Earth radii, planets stop being rock and start being small versions of
Neptune, with atmospheres thousands of kilometres deep and no ground under them. That line comes
from Rogers, *The Astrophysical Journal* **801**, 41 (2015) — "Most 1.6 Earth-radius planets are
not rocky", reference checked 2026-08-31. It is a convention, like the 273–373 K window, and worth
holding loosely.

So a planet inside the temperature window falls into one of three cases, and `None` makes the third
one unavoidable: rocky, too big to be rocky, or the archive never measured its radius.

### ✏️ Your turn 5

Loop over the usable planets. For each one, call `check_planet` with albedo
`0.0` — the same "reflecting nothing" pass the class started with — and collect every temperature
into a list called `all_temps`, which the next cell plots.

When the verdict is `"liquid water possible"`, add one to `n_window`, and then sort that planet
into one of three:

- `usable_radii[i] is None` — add one to `unknown_radius`;
- `usable_radii[i] < 1.6` — append the planet's name to `rocky_names`;
- anything else — add one to `too_big`.

Then print the four counts, and print each rocky candidate's name, temperature and radius, one per
line. You will need that printout for the homework.

**Use these names**, because the self-check looks for them: `all_temps`, `n_window`,
`unknown_radius`, `rocky_names` and `too_big`.

In [ ]:
# ← your answer here


In [ ]:
assert len(all_temps) == len(usable_names), "all_temps needs one entry per planet, not per hit"
assert unknown_radius > 0, "unknown_radius never moved — add one to it inside the `is None` branch"
print(f"✓ the survey — {n_window} planets in the window: {len(rocky_names)} rocky, "
      f"{too_big} too big, {unknown_radius} with no measured radius")

102 of the 189 have no published radius at all. That is not a rounding detail: it is more than half
the answer, and it is the difference between "there are 12 candidates" and "there are 12 candidates
and 102 unknowns". A test that had silently counted the unknowns as "not rocky" would have reported
the same 12 and looked far more confident than the data allows.

The figure below is every one of the three thousand temperatures, with the window marked.

In [ ]:
plt.hist(all_temps, bins=250)
plt.axvline(273, color="0.4")
plt.axvline(373, color="0.4")
plt.text(323, 105, "liquid water window", ha="center")   # ha= centres both on the band
plt.text(323, 94, "↓", ha="center")
plt.xlim(0, 2500)
plt.ylim(0, 115)                       # room above the bars for the label
plt.xlabel("equilibrium temperature at albedo 0 (K)")
plt.ylabel("number of planets")
plt.title(f"Every planet with the three numbers (n = {len(all_temps)})")
plt.show()

Most known planets are far hotter than the window, because a planet close to its star is the
easiest kind to find — a bias in the catalogue, not in the galaxy.

Now the interesting part. Astronomers keep an informal list of the planets thought most likely to
be habitable, and it is short: the outer TRAPPIST-1 planets, Proxima Centauri b, TOI-700 d,
Kepler-186 f, Kepler-442 b. The cell below asks our test about those, and about two more that are
not on it.

In [ ]:
famous = ["TRAPPIST-1 c", "TRAPPIST-1 e", "TRAPPIST-1 f", "TRAPPIST-1 g",
          "Proxima Cen b", "TOI-700 d", "Kepler-186 f", "Kepler-442 b", "K2-18 b"]
for i in range(len(usable_names)):
    if usable_names[i] in famous:
        temperature, verdict = check_planet(usable_star_temps[i], usable_star_radii[i],
                                            usable_distances[i], 0.0)
        print(f"{usable_names[i]}: {round(temperature, 1)} K, radius {usable_radii[i]} - {verdict}")

It rejects almost all of them. TRAPPIST-1 e, f and g, Proxima Cen b, TOI-700 d, Kepler-186 f and
Kepler-442 b all come out too cold, between 197.4 K and 267.8 K, on planets whose whole claim to
interest is that they might have air. That is the same failure the test made on Earth — but not
the same number, and the difference is worth being careful about. Every figure on that line is an
albedo-0 figure, and Earth at albedo 0 is 278.3 K and *passes*; it was Earth's own albedo, 0.306,
that pushed it down to 254.0 K. Two numbers computed at two different albedos are not comparable,
and the temptation to line them up anyway is exactly what makes this test easy to misread. What
does repeat, at either albedo, is the reason: a formula that knows nothing about air, asked about
worlds whose whole interest is their air. (Proxima Cen b prints its radius as `None`: it was found
by watching its star wobble, and that method never measures a size.)

It accepts two. TRAPPIST-1 c comes out at 339.9 K, hotter than any of its siblings we looked at,
and K2-18 b at 282.7 K — and K2-18 b is 2.37 Earth radii, a size our own radius branch had already
counted as too big to be rock, so the verdict string and the radius disagree about the same planet.
K2-18 b is also the planet the field has argued over hardest in recent years, water vapour in its
atmosphere and then a series of contested readings of its JWST spectrum, so "liquid water possible"
is the one verdict on that line nobody should read as a finding.

That accept-and-reject set is worth staring at, because it is strange without being random. Look
back at the twelve rocky candidates your loop printed: TRAPPIST-1 d, TOI-700 e, Kepler-438 b and K2-72 e are
all planets the field itself puts forward, so the test is not picking out some different set of
worlds from the ones astronomers care about. What it is doing is sorting **within** those systems
by temperature alone, keeping the warmer planet and dropping the cooler one — TRAPPIST-1 c and d
in, e, f and g out. That is the only thing a bare rock with no air can be sorted by, and it is
exactly the sorting that put Earth on the wrong side of the line.

## The question, answered

Reflecting nothing, the test accepts 189 planets — 12 of them small enough to be rock — and it
accepts Earth at 278.3 K and Venus at 327.3 K together. At each world's measured albedo it rejects
Earth at 254.0 K, Venus at 226.7 K and Mars at 209.8 K alike. It rejects Earth because equilibrium
temperature is the temperature of a bare rock in sunlight, and Earth is not one: 34.0 K of what
Earth has comes from its air, and 510.3 K of what Venus has comes from its own. The test is not
broken. It is answering a narrower question than the one we asked it, and the gap between the two
is the greenhouse effect.

## Week 2 summary

**The question.** Which of these worlds could have liquid water — and why does your test reject Earth?

### What to remember

| | |
|---|---|
| **1** | Equilibrium temperature ignores atmospheres: run the test one way and it accepts Venus alongside Earth, run it the other and it rejects both. It never gets the answer right. |
| **2** | The greenhouse effect arrives as a measured discrepancy, not a definition. |
| **3** | A function is a question you can ask many times without retyping it. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Accumulator pattern** | Set a counter to 0 before the loop starts and add to it inside; same idea for a list — start it empty and append inside. |
| **Equilibrium temperature** | The temperature a planet would sit at if starlight were the only thing heating it and it had no air. |
| **Albedo** | The fraction of the starlight falling on a world that it reflects straight back into space; only the rest is absorbed and turned into heat. |
| **The greenhouse effect** | The difference between the temperature a world would have with no air and the temperature it actually has. |

### Code you met this week

| Function | What it does |
|---|---|
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `list.append(x)` | add one item to the end of a list |
| `if / elif / else` | choose what to do, based on a comparison |
| `and / or / not` | combine two conditions into one |
| `abs(x)` | the size of a number, ignoring its sign |
| `x in things` | true when that value is somewhere in the list |
| `None` | Python's word for "nothing here" — you cannot compare it with a number |
| `plt.axvline(x)` | a vertical line at x, to mark a boundary on a plot |
| `plt.text(x, y, "Venus")` | write a word at a point on the axes |
| `def name(a, b):` | write the recipe once and give it a name; the names in brackets are what it needs |
| `return value` | hand one value back to whoever called the function |
| `a docstring` | triple-quoted text as the first thing inside a function, saying what it does |
| `return a, b` | hand back two values at once; catch them with a, b = f(...) |
| `help(f)` | print a function's docstring, which is why writing one pays |

## Homework

Three parts, on the same archive and the same tools. Part 1 is the one everybody finishes; part 3
is the one that takes thinking rather than typing.

Run the cell below first if you have restarted the kernel — after the setup cell at the top, it
rebuilds everything parts 1 and 2 use: the two constants, `equilibrium_temperature`, and the six
`usable_` lists from section 4. The one thing it cannot rebuild is `check_planet`, because that
one is your own answer to Your turn 4; re-run that cell too and part 1 will find it.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
sun_temp = 5772
sun_radius_au = 0.00465047


def equilibrium_temperature(star_temp, star_radius, distance, albedo):
    """the temperature starlight alone would hold a planet at, in kelvin"""
    return star_temp * (star_radius * sun_radius_au / (2 * distance)) ** 0.5 * (1 - albedo) ** 0.25


usable_names = []
usable_star_temps = []
usable_star_radii = []
usable_distances = []
usable_radii = []
usable_archive_temps = []
for i in range(len(planet_names)):
    if star_temps[i] is not None and star_radii[i] is not None and distances[i] is not None:
        usable_names.append(planet_names[i])
        usable_star_temps.append(star_temps[i])
        usable_star_radii.append(star_radii[i])
        usable_distances.append(distances[i])
        usable_radii.append(planet_radii[i])
        usable_archive_temps.append(archive_temps[i])

# Re-run your own check_planet cell (Your turn 4) as well: part 1 calls it, and it is not
# repeated here, because it is the answer to a question you were asked.

### ✏️ Your turn 6

**What kind of stars are these?**

Class asked which planets could hold liquid water and never asked what they orbit. A star's
temperature is the first number the formula takes, and it has been sitting in `usable_star_temps`
all along.

It carries more than it looks. A star much cooler than the Sun is also much smaller and fainter,
so a planet warm enough for liquid water has to sit very close in — close enough to be locked with
one face to its star for good, and close enough that a stellar flare arrives at full strength.
Whether these candidates orbit Sun-like stars or cool ones is the most consequential thing about
the list, and nobody has looked yet.

Loop over the usable planets and call your own `check_planet` with albedo `0.0`, exactly as
section 4 did. For every planet whose verdict comes back `"liquid water possible"` **and** whose
radius is not `None` **and** is below 1.6 — the same rocky candidates section 4 counted — append
its name to `candidate_names` and its star's temperature to `candidate_star_temps`. Two lists
growing together inside the same `if`.

Then print: how many candidates there are; how many of them orbit a star cooler than the Sun's
5772 K, counted into `cooler_than_sun` with a second loop; and the value of the coolest star in
the set, `coolest`, together with the planet it belongs to. `min(candidate_star_temps)` gives you
the coolest value and `.index(...)` gives you its position, exactly as last week's `max` and
`.index` found the largest earthquake and then its place name.

One thing about `.index` that last week never made you face: it hands back the **first** position
holding that value and stops looking. Two planets of the same system orbit the same star, so they
carry the same star temperature to the last decimal — and if that star is the coolest one, `.index`
will name one of those planets and say nothing about the other. So finish by looping once more over
the candidates and printing each name beside its star's temperature, one per line. That printout is
what tells you whether the planet `.index` picked was alone, and it is the one to read before you
believe any single line above it.

Then, in two or three sentences in the last cell of this part, say what kind of stars these
candidates orbit: quote `cooler_than_sun` out of your total and name the planet `.index` gave you
for `coolest`, and then say what a star that cool forces on a planet warm enough for liquid water,
and whether the formula knows anything about it.

**Use these names**, because the self-check looks for them: `candidate_names`,
`candidate_star_temps`, `cooler_than_sun`, `coolest` and `check_planet`.

In [ ]:
# ← your answer here


In [ ]:
assert len(candidate_names) == len(candidate_star_temps), \
    "the two lists must grow together — append to both inside the same if"
assert len(candidate_names) < 100, "that is everything in the window; the radius test is missing"
print(f"✓ Homework 1 — {len(candidate_names)} candidates, {cooler_than_sun} around stars "
      f"cooler than the Sun, coolest {coolest} K")

*(Double-click this cell and replace this line with your answer.)*

### ✏️ Your turn 7

**Move one knob, and count what moves.**

Both of the choices class made were arguable. Pick **one** of these two changes, and only one.

- **Option A — the mirror.** Venus reflects 0.770 of the light that reaches it, and there is no
  reason a distant rocky planet should reflect nothing. Set `albedo = 0.770`, leave `low = 273` and
  `high = 373`.
- **Option B — the floor.** Air can only warm a planet, never cool it, so the albedo-0 equilibrium
  temperature is a floor rather than an estimate: a world 20 K below freezing on paper could still
  have liquid water underneath a thick atmosphere. Set `albedo = 0.0`, `low = 250` and
  `high = 373`.

Then rerun part 1's loop with your three values in place of the fixed ones, into a list called
`new_candidates`, and report what moved: how many candidates there are now, how many of part 1's
`candidate_names` are still on the list, how many dropped off and how many are new. `name in
candidate_names` is true when that name is somewhere in the list. (The first number stands on its
own; the other three need part 1 to have run.)

Say which option you took in a comment on the first line.

Then, in two or three sentences in the last cell of this part, say what your four numbers mean for
the class's answer: quote how many candidates you started with and how many you have now, and say
whether the twelve rocky candidates class found are a property of those twelve planets or of the
two numbers class happened to pick.

**Use these names**, because the self-check looks for them: `albedo`, `low`, `high`,
`new_candidates` and `stayed`.

In [ ]:
# ← your answer here


In [ ]:
assert albedo == 0.770 or low == 250, "neither knob moved — set up option A or option B"
assert len(new_candidates) != len(candidate_names), "the count did not move; check what you changed"
print(f"✓ Homework 2 — albedo {albedo}, window {low}-{high} K: "
      f"{len(new_candidates)} candidates, {stayed} of part 1's still there")

*(Double-click this cell and replace this line with your answer.)*

### ✏️ Your turn 8

**One planet you do not believe.**

Your turn 5's printout lists every planet this week's test calls a rocky candidate, with its
temperature and its radius. Pick **one** of them that you do not believe could have liquid water,
and make the case against it in two or three sentences, in the cell below.

Quote real numbers, not impressions: that planet's own temperature and radius from the printout, and
how far the same test was wrong about Earth and about Venus — both of those differences were printed
in section 3. Then finish with one sentence naming what the test does not know about your planet
that would settle it.

There is no single right answer. There is a defensible one with three numbers in it.

*(Double-click this cell and replace this line with your answer.)*